# 17 — Paper Figures: Pipeline Progression

Two panels. (a) walks GRU's pipeline stage by stage, plotting TSS and POD
together on one axis. Both are prevalence-invariant skill scores, so they
are directly comparable at the same scale; showing them together makes the
point that they move in lockstep, both peaking after Tomek Links cleaning
and both collapsing at the final over-sampling stage, unlike Bias, HSS2, and
Accuracy (discussed via Table 8 instead of a figure), which drift the
opposite way over the same stages.

(b) is a plain bar chart of each classifier's own best full-pipeline TSS
(the same configurations as Table 8 / Figure 6a), sorted low to high.

**Reads:** `./results/*.txt`. **Requires:** notebook 14 run first.
**Writes:** `./paper/figures/pipeline_progression.pdf` / `.png`.


## 1. Load and Preview

In [3]:
# ══════════════════════════════════════════════════════════════
# GRU's own stage-by-stage progression (its winning pipeline stops at
# "+ Clean" -- the remaining stages are shown to make the point, not
# because they are recommended).
# ══════════════════════════════════════════════════════════════

STAGES = [
    ("Raw",                 "nonorm"),
    ("+ Normalize",         "hybrid"),
    ("+ Clean",             "hybrid_tomek"),
    ("+ RUS 8000",          "tomek_nonsep_8000"),
    ("+ OS\n(ADASYN 8000)", "tomek_rus8000_adasyn8000"),
]

print(f"{'stage':<24} {'mean TSS':>9} {'mean POD':>10}")
for label, key in STAGES:
    tss = load_col("gru", key, COLUMN["tss"])
    rec = load_col("gru", key, COLUMN["recall"])
    print(f"{label!r:<24} {tss.mean():>9.3f} {rec.mean():>10.3f}")

# Raw vs best-pipeline TSS, one row per classifier
BEST_CONFIG = {
    "GRU": "hybrid_tomek",
    "PatchTST": "tomek_rus8000_adasyn8000",
    "SVM": "minmax_all",
    "InceptionTime": "tomek_nonsep_8000",
}
print("\nRaw vs. final, per classifier:")
for clf, key in BEST_CONFIG.items():
    slug = {v: k for k, v in CLASSIFIER_LABELS.items()}[clf]
    raw = load_col(slug, "nonorm").mean()
    fin = load_col(slug, key).mean()
    print(f"  {clf:<14} raw={raw:>7.3f}  final={fin:>7.3f}  lift={fin - raw:>+7.3f}")


stage                     mean TSS   mean POD
'Raw'                       -0.190      0.338
'+ Normalize'                0.609      0.779
'+ Clean'                    0.671      0.794
'+ RUS 8000'                 0.647      0.779
'+ OS\n(ADASYN 8000)'        0.251      0.265

Raw vs. final, per classifier:
  GRU            raw= -0.190  final=  0.671  lift= +0.860
  PatchTST       raw=  0.502  final=  0.623  lift= +0.121
  SVM            raw=  0.148  final=  0.505  lift= +0.357
  InceptionTime  raw=  0.013  final=  0.404  lift= +0.391


## Figure — Stage Progression and Total Lift

In [4]:
# FIG -- pipeline progression: GRU stage-by-stage (TSS + POD), and best pipeline per classifier
fig, axes = plt.subplots(1, 2, figsize=(7.16, 2.55),
                         gridspec_kw=dict(width_ratios=[1.15, 1.0]))

# ---------------------------------------------------------------
# (a) GRU stage progression: TSS and POD together on one axis. Both are
# prevalence-invariant, so they belong on the same scale -- and they move
# together, both peaking after Tomek Links cleaning and both collapsing
# at the final over-sampling stage (unlike Bias/HSS2/Accuracy, which move
# the opposite way over the same stages; see Table 8).
# ---------------------------------------------------------------
ax = axes[0]
xs = np.arange(len(STAGES))
tss_vals = np.array([load_col("gru", k, COLUMN["tss"]).mean() for _, k in STAGES])
pod_vals = np.array([load_col("gru", k, COLUMN["recall"]).mean() for _, k in STAGES])

ax.plot(xs, tss_vals, "-o", color=TEAL, lw=1.3, ms=3.6, mfc="white", mew=1.0,
        zorder=4, label="TSS")
ax.plot(xs, pod_vals, "-s", color=BLUE, lw=1.1, ms=3.2, mfc="white", mew=0.9,
        zorder=4, label="POD")
# Endpoints only -- the middle three stages pack TSS and POD close enough
# together that per-point labels there collide; Table 8 has the exact values.
for xi, v in [(xs[0], tss_vals[0]), (xs[-1], tss_vals[-1])]:
    ax.text(xi, v + 0.06, f"{v:.2f}", ha="center", va="bottom", fontsize=5.8,
            color=TEAL, zorder=6,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.85, pad=0.5))
for xi, v in [(xs[0], pod_vals[0]), (xs[-1], pod_vals[-1])]:
    ax.text(xi, v - 0.07, f"{v:.2f}", ha="center", va="top", fontsize=5.8,
            color=BLUE, zorder=6,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.85, pad=0.5))
ax.axhline(0, color="0.75", lw=0.5, zorder=1)
ax.set_xticks(xs)
ax.set_xticklabels([s[0] for s in STAGES], fontsize=5.6)
ax.set_ylim(-0.32, 0.95)
finish(ax, ylab="Score")
ax.legend(frameon=False, loc="upper left", handlelength=1.2, handletextpad=0.4,
          borderpad=0.1, labelspacing=0.25, fontsize=6.2)
ax.set_title("(a)  GRU: stage by stage (TSS and POD)", loc="left", fontsize=7.0, pad=4)

# ---------------------------------------------------------------
# (b) Best full-pipeline TSS per classifier (same configurations as
# Table 8 / Figure 6a), sorted low to high.
# ---------------------------------------------------------------
ax = axes[1]
rows = []
for clf, key in BEST_CONFIG.items():
    slug = {v: k for k, v in CLASSIFIER_LABELS.items()}[clf]
    v = load_col(slug, key)
    rows.append((clf, v))
rows.sort(key=lambda r: r[1].mean())

for i, (clf, v) in enumerate(rows):
    col = TEAL if clf == "GRU" else "0.62"
    ax.barh(i, v.mean(), height=0.6, color=col, edgecolor="none", zorder=3)
    ax.scatter(v, np.full_like(v, i, dtype=float), s=16, color="0.15", lw=0, zorder=5)
    ax.text(v.mean() + 0.02, i, f"{v.mean():.3f}", va="center", ha="left",
            fontsize=6.2, color="0.0", zorder=6,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.82, pad=0.5))
ax.set_yticks(range(len(rows)))
ax.set_yticklabels([r[0] for r in rows], fontsize=6.4)
ax.set_xlim(0, 0.82)
finish(ax, xlab="Test TSS", grid="x")
ax.set_title("(b)  Best pipeline per classifier", loc="left", fontsize=7.0, pad=4)

fig.subplots_adjust(left=0.09, right=0.97, top=0.90, bottom=0.16, wspace=0.45)
fig.savefig(f"{FIG_DIR}/pipeline_progression.pdf")
fig.savefig(f"{FIG_DIR}/pipeline_progression.png", dpi=340)
plt.show()
print("Saved pipeline_progression.pdf/png")


Saved pipeline_progression.pdf/png


/var/folders/fx/gjhbmrbj5jn295_9wrqpbsv80000gn/T/ipykernel_41188/2623761387.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
